In [2]:
# 1. Imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

import os
from dotenv import load_dotenv
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# 2. Setup LLM and Embeddings

llm = ChatOpenAI(temperature=0, model="gpt-4.1-mini")
embedding = OpenAIEmbeddings()

# 3. Load your document
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 4. Split it into chunks
splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
chunks = splitter.split_text(raw_text)
documents = [Document(page_content=chunk) for chunk in chunks]

# 5. Compress the documents using LLMChainExtractor
compressor = LLMChainExtractor.from_llm(llm)
compressed_docs = compressor.compress_documents(documents, query="Summarize the key points")

print("Compressed Summary:")
for doc in compressed_docs:
    print("-", doc.page_content)

# 6. Now ask a question about the compressed content
# (replaces the deprecated LLMChain with an LCEL pipe: prompt | llm | output-parser)
qa_prompt = PromptTemplate.from_template(
    "Given the context below, answer the question:\n\nContext:\n{context}\n\nQuestion: {question}"
)
qa_chain = qa_prompt | llm | StrOutputParser()

response = qa_chain.invoke({
    "context": "\n".join([doc.page_content for doc in compressed_docs]),
    "question": "What is the main idea of the document?"
})

print("\n Answer to your question:")
print(response)

Compressed Summary:
- LangChain is a framework for building applications with LLMs.LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&A, and AI workflow

 Answer to your question:
The main idea of the document is to introduce LangChain as a framework for building applications with large language models (LLMs), highlighting its creator, key features like RAG, agents, memory, and tools, and its common use cases such as chatbots, document question answering, and AI workflows.
